In [ ]:
pip install datasets

In [ ]:
import os
import random

import numpy as np
import torch
import tqdm
from datasets import load_dataset
from gpt import AndersenGPT
from torch import nn
from torch.utils.data import DataLoader
from transformers import AutoTokenizer

# -----------------------
# Hyperparameters & Configs
# -----------------------
# Model
EMBED_DIM = 768
NUM_HEADS = 12
NUM_LAYERS = 12
MAX_SEQ_LEN = 1024  # Context length
PRETRAINED_TOKENIZER = "gpt2"
POS_ENC = "learnable"  # Options: learnable, fixed
# Training
START_FROM_PRETRAINED_GPT2_CHECKPOINT = True
BATCH_SIZE = 3
NUM_EPOCHS = 10
LR = 1e-4
WARMUP_STEPS = 625
WEIGHT_DECAY = 1e-4
GRADIENT_CLIPPING = 1.0
DATASET_NAME = "monurcan/andersen_fairy_tales"
MODEL_SAVE_PATH = "checkpoints"


# -----------------------
# Utility functions
# -----------------------
def set_seed(seed=1):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True


# -----------------------
# Data Preparation
# -----------------------
def prepare_data_iter(tokenizer, max_seq_len=512, batch_size=16):
    """
    Loads the dataset, tokenizes each story without truncation, and then
    splits the long token sequence into many training samples (chunks).
    Each chunk is of length max_seq_len+1 tokens so that when you shift,
    the inputs have length max_seq_len and the labels have length max_seq_len.
    """
    # Load the dataset
    dataset = load_dataset(DATASET_NAME)
    train_dataset = dataset["train"]
    test_dataset = dataset["validation"]

    def tokenize_and_chunk(examples):
        # Tokenize without truncation/padding.
        outputs = tokenizer(examples["story"], add_special_tokens=True)
        all_chunks = {"input_ids": []}
        for tokens in outputs["input_ids"]:
            # You can use non-overlapping chunks (or add overlap if desired).
            # Here we use a non-overlapping strategy.
            # We want each chunk to have (max_seq_len+1) tokens to allow shifting.
            for i in range(0, len(tokens), max_seq_len + 1):
                chunk = tokens[i : i + max_seq_len + 1]
                # Only keep chunks that have at least 2 tokens (needed for input and label)
                if len(chunk) > 1:
                    all_chunks["input_ids"].append(chunk)
        return all_chunks

    # Tokenize and chunk the train and validation splits.
    # We remove all original columns so that each output is just our chunk.
    train_dataset = train_dataset.map(
        tokenize_and_chunk,
        batched=True,
        remove_columns=train_dataset.column_names,
    )
    test_dataset = test_dataset.map(
        tokenize_and_chunk,
        batched=True,
        remove_columns=test_dataset.column_names,
    )

    # The previous mapping produces a nested list (one list per original example).
    # Flatten the dataset so that each element is a single training sample (i.e. a chunk).
    train_dataset = train_dataset.flatten()
    test_dataset = test_dataset.flatten()

    def collate_fn(batch):
        # Each element of batch is a dict: {"input_ids": chunk}
        # Convert to tensors.
        input_ids = [
            torch.tensor(example["input_ids"], dtype=torch.long) for example in batch
        ]
        # Pad sequences in the batch to the maximum length (which might be less than max_seq_len+1)
        input_ids = torch.nn.utils.rnn.pad_sequence(
            input_ids, batch_first=True, padding_value=tokenizer.pad_token_id
        )
        # For next-token prediction, the model input is everything except the last token
        # and the target (labels) is everything except the first token.
        inputs = input_ids[:, :-1]
        labels = input_ids[:, 1:]
        return inputs, labels

    train_iter = DataLoader(
        train_dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn
    )
    test_iter = DataLoader(
        test_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn
    )
    return train_iter, test_iter


# -----------------------
# Main Training Loop
# -----------------------
def main(
    embed_dim=EMBED_DIM,
    num_heads=NUM_HEADS,
    num_layers=NUM_LAYERS,
    num_epochs=NUM_EPOCHS,
    pos_enc=POS_ENC,
    dropout=0.0,
    fc_dim=None,
    batch_size=BATCH_SIZE,
    lr=LR,
    warmup_steps=WARMUP_STEPS,
    weight_decay=WEIGHT_DECAY,
    gradient_clipping=GRADIENT_CLIPPING,
    max_seq_len=MAX_SEQ_LEN,
):
    # Use a pretrained tokenizer from Hugging Face.
    tokenizer = AutoTokenizer.from_pretrained(PRETRAINED_TOKENIZER)
    # Ensure the tokenizer uses a padding token.
    # GPT2 has no pad token by default, so we assign the EOS token as pad_token.
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    # Get training and validation iterators.
    train_iter, test_iter = prepare_data_iter(
        tokenizer, max_seq_len=max_seq_len, batch_size=batch_size
    )
    vocab_size = tokenizer.vocab_size  # use the tokenizer's vocab size

    # Instantiate the GPT model.
    model = AndersenGPT(
        embed_dim=embed_dim,
        num_heads=num_heads,
        num_layers=num_layers,
        max_seq_len=max_seq_len,
        pos_enc=pos_enc,
        dropout=dropout,
        fc_dim=fc_dim,
        num_tokens=vocab_size,
    )

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)

    if START_FROM_PRETRAINED_GPT2_CHECKPOINT:
        model.load_state_dict(torch.load("gpt2_pretrained.pt"))

    # Define the loss function for language modeling.
    # We ignore the padding token (which is now set to eos_token if needed)
    loss_function = nn.CrossEntropyLoss(ignore_index=tokenizer.pad_token_id)

    # Set up optimizer and learning rate scheduler.
    optimizer = torch.optim.AdamW(
        lr=lr, params=model.parameters(), weight_decay=weight_decay
    )
    scheduler = torch.optim.lr_scheduler.LambdaLR(
        optimizer, lambda i: min(i / warmup_steps, 1.0)
    )

    best_val_loss = float("inf")

    # Training loop
    for epoch in range(num_epochs):
        print(f"\nEpoch {epoch + 1}/{num_epochs}")
        model.train()
        train_loss = 0.0
        num_train_tokens = 0

        for batch in tqdm.tqdm(train_iter, desc="Training"):
            optimizer.zero_grad()
            inputs, labels = batch
            inputs = inputs.to(device)
            labels = labels.to(device)

            # Forward pass: outputs has shape (batch_size, seq_len, vocab_size)
            logits = model(inputs)
            # Flatten logits and labels for computing cross entropy loss.
            logits = logits.view(-1, vocab_size)
            labels = labels.view(-1)
            loss = loss_function(logits, labels)
            loss.backward()
            if gradient_clipping > 0.0:
                nn.utils.clip_grad_norm_(model.parameters(), gradient_clipping)
            optimizer.step()
            scheduler.step()

            train_loss += loss.item() * labels.numel()
            num_train_tokens += labels.numel()

        avg_train_loss = train_loss / num_train_tokens
        print(
            f"  Training Loss: {avg_train_loss:.4f} | Perplexity: {np.exp(avg_train_loss):.4f}"
        )

        # Validation loop
        model.eval()
        val_loss = 0.0
        num_val_tokens = 0
        with torch.no_grad():
            for batch in tqdm.tqdm(test_iter, desc="Validation"):
                inputs, labels = batch
                inputs = inputs.to(device)
                labels = labels.to(device)
                logits = model(inputs)
                logits = logits.view(-1, vocab_size)
                labels = labels.view(-1)
                loss = loss_function(logits, labels)
                val_loss += loss.item() * labels.numel()
                num_val_tokens += labels.numel()

        avg_val_loss = val_loss / num_val_tokens
        print(
            f"  Validation Loss: {avg_val_loss:.4f} | Perplexity: {np.exp(avg_val_loss):.4f}"
        )

        # Save the model with the best validation loss
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save(model.state_dict(), os.path.join(MODEL_SAVE_PATH, "best.pt"))
            print(f"Model saved with best validation loss: {best_val_loss:.4f}")

    # Save the final model after training is complete
    torch.save(model.state_dict(), os.path.join(MODEL_SAVE_PATH, "final.pt"))
    print(f"Final model saved to {MODEL_SAVE_PATH}/final.pt")


if __name__ == "__main__":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Model will run on {device}")
    set_seed(seed=1)

    # Create the checkpoint directory if it doesn't exist
    if not os.path.exists(MODEL_SAVE_PATH):
        os.makedirs(MODEL_SAVE_PATH)

    main()


Model will run on cuda


<ipython-input-4-520fe94a6be7>:168: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("gpt2_pretrained.pt"))



Epoch 1/10


Training: 100%|██████████| 184/184 [03:06<00:00,  1.02s/it]


  Training Loss: 3.2317 | Perplexity: 25.3233


...

  Validation Loss: 4.7871 | Perplexity: 119.9587

Epoch 10/10


Training: 100%|██████████| 184/184 [03:06<00:00,  1.01s/it]


  Training Loss: 0.4959 | Perplexity: 1.6419


Validation: 100%|██████████| 6/6 [00:01<00:00,  3.19it/s]


  Validation Loss: 5.2045 | Perplexity: 182.0945
Final model saved to checkpoints/final.pt


In [ ]:
import torch
from gpt import AndersenGPT
# from train import (
#     EMBED_DIM,
#     MAX_SEQ_LEN,
#     MODEL_SAVE_PATH,
#     NUM_HEADS,
#     NUM_LAYERS,
#     POS_ENC,
#     PRETRAINED_TOKENIZER,
# )
from transformers import AutoTokenizer


def generate_text(model, tokenizer, prompt, max_gen_len=500, device="cpu"):
    """
    Given a prompt string, generate a continuation using greedy decoding.
    The prompt is encoded using the pretrained tokenizer.
    """
    # Encode the prompt (returns a list of token ids)
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)

    for _ in range(max_gen_len):
        ####################### insert code here #####################################################
        # Ensure we work with the last MAX_SEQ_LEN tokens if the sequence gets too long.
        # If the sequence is longer than MAX_SEQ_LEN, keep only the last MAX_SEQ_LEN tokens.
        # Check if input_ids is too long, if it is crop it
        if input_ids.size(1) > MAX_SEQ_LEN:
            input_ids = input_ids[:, -MAX_SEQ_LEN:]

        # print("input size", end=" ")
        # print(input_ids.size())

        # Forward pass: get logits for all tokens in the sequence.
        logits = model(input_ids)

        # print("logit size", end=" ")
        # print(logits.size())

        # Get the logits for the last token only: shape [batch_size, vocab_size]
        next_token_logits = logits[:, -1, :]

        # print("next_token size", end = " ")
        # print(next_token_logits.size())

        # You will implement two strategies for generating the next token:
        strategy = "sampling"
        if strategy == "greedy":
            # Greedy: choose the token with highest probability.
            next_token_id = torch.argmax(next_token_logits, dim=-1, keepdim=True)
        elif strategy == "sampling":
            # Multinomial Sampling: Sample from the probability distribution.
            # The temperature parameter controls the randomness of the sampling.
            temperature = 1
            probabilities = torch.softmax(next_token_logits/temperature, dim = 1) # softmax with temperature
            next_token_id = torch.multinomial(probabilities, num_samples=1)

        # Append predicted token to input_ids. Concatenate
        input_ids = torch.cat([input_ids, next_token_id], dim=1)

        # Stop early if the model generates the EOS token.
        # Check if next_token_id == tokenizer.eos_token_id
        # If next_token is end of sentence token, it should stop

        if next_token_id.item() == tokenizer.eos_token_id:
            break
        ################################################################################################

    # Decode the full sequence to text.
    output_text = tokenizer.decode(input_ids.squeeze(), skip_special_tokens=True)
    return output_text


@torch.no_grad()
def main():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Loading model on {device} ...")

    # Load the same pretrained tokenizer used during training.
    tokenizer = AutoTokenizer.from_pretrained(PRETRAINED_TOKENIZER)

    # GPT2 does not have a PAD token by default; set it to the EOS token.
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    # Instantiate the GPT-style model with the same hyperparameters as during training.
    model = AndersenGPT(
        embed_dim=EMBED_DIM,
        num_heads=NUM_HEADS,
        num_layers=NUM_LAYERS,
        max_seq_len=MAX_SEQ_LEN,
        pos_enc=POS_ENC,
        dropout=0.0,
        fc_dim=None,
        num_tokens=tokenizer.vocab_size,
    ).to(device)

    # Load the model checkpoint.
    state_dict = torch.load("checkpoints/final.pt", map_location=device)
    model.load_state_dict(state_dict)
    model.eval()
    print("Model loaded successfully.\n")

    print("Enter a prompt and the model will generate a continuation.")
    print("Type 'quit' or 'exit' to stop.\n")
    while True:
        prompt = input("Prompt: ").strip()  # Stripping is for tokenization weirdness
        if prompt.lower() in ["quit", "exit"]:
            break
        generated_text = generate_text(
            model, tokenizer, prompt, max_gen_len=500, device=device
        )
        print("\n--- Generated Text ---")
        print(generated_text)
        print("----------------------\n")


if __name__ == "__main__":
    main()


Loading model on cuda ...


<ipython-input-5-d7c1b9dba782>:99: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load("checkpoints/final.pt", map_location=device)


Model loaded successfully.

Enter a prompt and the model will generate a continuation.
Type 'quit' or 'exit' to stop.

Prompt: Mermaids are known for

--- Generated Text ---
Mermaids are known for their
treasure. They carry a bundle, with their stories, tales, and
photographs, and they have really more information than the
wholesale world. They are not vain, but inwardly they hide the good
story after the outward way; and in this way the people of the
river run themselves into the sea, and lie asleep at the Corellia
bundle-end, where they can know everything.



During the night, the fishermen lure a large white swan in their
hands, and tie him up in this manner: he is like a glove out in the
water. Deep water is close by, but he is not touched till he is placed
against the wall. At about sunrise the bird flies away, for he is
already wet, and as he flies away he makes his appearance behind the nettles.
Once upon a time there were mermaids lie in the water, who knew
about the nest with

KeyboardInterrupt: Interrupted by user